Les codes ci-dessous ont été réalisé à l'aide d'IA générative.

Import de nos bibliothèques

In [ ]:
import pandas as pd
import rasterio
import numpy as np
from pathlib import Path
import subprocess
import matplotlib.pyplot as plt
from rasterio.plot import show

Méthode de subsampling

In [ ]:
fichier = "/Users/THEO/Documents/Projet/PROJET_kaliptus/petauroides/0005928-260928105237408.csv"

df = pd.read_csv(fichier, sep='\t', low_memory=False)

# Définir la taille de la grille 
cell_size = 0.16667

df = df.dropna(subset=['decimalLongitude', 'decimalLatitude'])

df['grid_x'] = (df['decimalLongitude'] / cell_size).astype(int)
df['grid_y'] = (df['decimalLatitude'] / cell_size).astype(int)

# Ne garder qu'un point par cellule
df_clean = df.drop_duplicates(subset=['grid_x', 'grid_y'])

# Exporter pour Maxent
df_clean[['species', 'decimalLongitude', 'decimalLatitude']].to_csv("petauroides.csv", index=False)

Choix des variables bioclimatiques que l'on va extraire de 2081 à 2100

In [ ]:
import rasterio
from pathlib import Path

future_tif = Path("C:/Users/THEO/Documents/Projet/PROJET_kaliptus/original_data/wc2.1_10m_bioc_ACCESS-CM2_ssp585_2081-2100.tif")
output_dir = Path("C:/Users/THEO/Documents/Projet/PROJET_kaliptus/old_data/BioClim variables/wc_biovar2080_tif")
output_dir.mkdir(parents=True, exist_ok=True)

# Correspondance entre le numéro de bande et le nom de la variable Bioclim
# Bande 1 = bio1, Bande 5 = bio5, etc.
target_bands = {
    1: "wc2.1_10m_bio_1",
    5: "wc2.1_10m_bio_5",
    6: "wc2.1_10m_bio_6",
    12: "wc2.1_10m_bio_12",
    14: "wc2.1_10m_bio_14",
    15: "wc2.1_10m_bio_15"
}

with rasterio.open(future_tif) as src:
    for band_num, var_name in target_bands.items():
        # Lecture de la bande spécifique
        data = src.read(band_num)
        
        # Copie des métadonnées et mise à jour pour 1 seule bande
        meta = src.meta.copy()
        meta.update(count=1)
        
        output_file = output_dir / f"{var_name}.tif"
        with rasterio.open(output_file, 'w', **meta) as dst:
            dst.write(data, 1)
        
        print(f"Extraction réussie : {var_name}.tif")

print("Toutes les variables bioclimatiques futures ont été extraites avec succès !")

Convertisseur de fichiers, passage de .tif à .asc

In [ ]:
# Dossier contenant vos fichiers .tif (ex: bio1.tif, bio5.tif, etc.)
input_dir = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\tif_input")
# Dossier où seront enregistrés les fichiers .asc
output_dir = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\asc_output")

output_dir.mkdir(parents=True, exist_ok=True)

NODATA_VAL = -9999

# Boucle sur tous les fichiers .tif du dossier
for input_tif in input_dir.glob("*.tif"):
    # Crée le nom du fichier de sortie .asc en gardant le même nom de base (ex: bio1.asc)
    output_asc = output_dir / f"{input_tif.stem}.asc"
    
    with rasterio.open(input_tif) as src:
        meta = src.meta.copy()

        # 1. Lire la bande 1 (vos fichiers extraits n'ont qu'une seule bande)
        data = src.read(1)

        # 2. Remplacer les NaN par une valeur nodata numérique
        data = np.where(np.isnan(data), NODATA_VAL, data)

        # 3. Configurer l'export pour le format ASCII Grid (AAIGrid)
        meta.update(
            driver="AAIGrid",
            count=1,
            dtype="float32",
            nodata=NODATA_VAL
        )

        # 4. Ecrire le fichier .asc
        with rasterio.open(output_asc, 'w', **meta) as dst:
            dst.write(data, 1)

    print(f"Conversion réussie : {input_tif.name} -> {output_asc.name}")

print("Toutes les conversions en .asc sont terminées avec succès !")

Projections MaxEnt après que tout les fichiers ont été converti en asc et que les data ont été subsampled

In [ ]:
java_exe = Path(r"C:\Program Files\Java\jre1.8.0_471\bin\java.exe")
maxent_jar = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\maxent\maxent.jar")

# Dossiers de données
env_present = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\bioclimatic_var_present")
env_future = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\bioclimatic_var_2080_ssp585")
occ_dir = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\occurences")

output_root = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\maxent_results")
output_root.mkdir(parents=True, exist_ok=True)

# Vos 3 espèces (attention à bien nommer vos fichiers CSV de la même manière)
species_list = ["koala", "eucalyptus", "petauroides"]
regularization_multipliers = [0.5, 1.0, 2.0] # Test de différents paramètres de régularisation

for species in species_list:
    occ_file = occ_dir / f"{species}.csv"
    
    for rm in regularization_multipliers:
        exp_name = f"{species}_rm{rm}"
        current_output = output_root / exp_name
        current_output.mkdir(exist_ok=True)
        
        cmd = [
            str(java_exe),
            "-mx2048m",
            "-jar", str(maxent_jar),
            f"samplesfile={occ_file}",
            f"environmentallayers={env_present}",
            f"projectionlayers={env_future}", # Permet de projeter directement sur le climat 2080 !
            f"outputdirectory={current_output}",
            f"betamultiplier={rm}",
            "autorun",
            "outputformat=logistic",
            "randomseed=true"
            # "replicas=5", # Validation croisée (folds) demandée par le sujet
            # "replicatype=crossvalidate"
        ]
        
        print(f"Exécution de MaxEnt pour : {exp_name}")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"-> Succès pour {exp_name}")
        else:
            print(f"-> Erreur sur {exp_name} : {result.stderr}")

print("Toutes les modélisations et projections 2080 sont terminées !")

Binarisation des données des 3 datasets après projection individuel ainsi que intersection

In [ ]:
# Dossier principal et sous-dossier rm1
base_dir = Path(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\maxent_results")

# Chemins vers tes fichiers .asc (adapte les noms si besoin selon tes sous-dossiers)
path_koala = base_dir / "koala_rm1.0" / "Phascolarctos_cinereus.asc"
path_euc = base_dir / "eucalyptus_rm1.0" / "Eucalyptus_viminalis.asc"
path_peta = base_dir / "petauroides_rm1.0" / "Petauroides_volans.asc"
output_raster = base_dir / "refuges_rm1_intersection.tif"

# 1. Ouvrir et lire les 3 rasters rm1
with rasterio.open(path_koala) as src_k, \
     rasterio.open(path_euc) as src_e, \
     rasterio.open(path_peta) as src_p:
     
    koala_arr = src_k.read(1)
    euc_arr = src_e.read(1)
    peta_arr = src_p.read(1)
    profile = src_k.profile

# 2. Binarisation (Seuil à 0.5)
threshold = 0.5
koala_bin = (koala_arr >= threshold).astype(int)
euc_bin = (euc_arr >= threshold).astype(int)
peta_bin = (peta_arr >= threshold).astype(int)

# 3. Intersection (ET logique : les zones où les 3 espèces coexistent sous rm1)
refuge_grid = (koala_bin * euc_bin * peta_bin)

# 4. Sauvegarder le résultat de l'intersection en GeoTIFF
profile.update(dtype=rasterio.int32, count=1, nodata=-9999)
with rasterio.open(output_raster, 'w', **profile) as dst:
    dst.write(refuge_grid.astype(rasterio.int32), 1)

# 5. Compter les pixels pour un premier aperçu de la surface
pixel_count = np.sum(refuge_grid == 1)
print(f"✅ Intersection rm1 exécutée avec succès !")
print(f"Nombre de pixels de refuges communs : {pixel_count}")

Création de map

In [ ]:
# Chemin vers ton fichier .asc de sortie MaxEnt
asc_file = r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\refuges_climatiques_mondial_2080.asc"
with rasterio.open(asc_file) as src:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Lire les données et masquer les valeurs négatives (NoData de MaxEnt)
    data = src.read(1)
    data_masked = np.ma.masked_where(data < 0, data)
    
    # Afficher la carte avec les coordonnées géographiques réelles (grâce à src.transform)
    show(data_masked, transform=src.transform, ax=ax, cmap='terrain', vmin=0, vmax=1)
    
    # --- AJOUT DE LA COLORBAR ---
    im = ax.get_images()[0]  # Récupère l'affichage raster de la carte
    fig.colorbar(im, ax=ax, label="Suitability (0 to 1)", shrink=0.7)
    
    plt.title("Refuges climatiques Elapid - 2080", fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    
    # Optionnel : réactiver les axes pour voir les frontières de l'Australie en coordonnées
    # plt.grid(True, linestyle='--', alpha=0.5)
    
    # Sauvegarder l'image propre pour ton rapport LaTeX
    plt.savefig(r"C:\Users\THEO\Documents\Projet\PROJET_kaliptus\img\refuges_climatiques_mondial_2080.png", dpi=300, bbox_inches='tight')
    plt.show()


